# Field deploy — ship a validated field as a queryable, calibrated column

Run **after** `field_factory.ipynb` and after the human label review. The quality gate is
explicit: this notebook refuses to run without a reviewed labels file, and prints the
validation numbers (incl. flip rate) before doing anything expensive.

Steps:
1. **Gate** — load reviewed labels, show flip rate + F1; deployment calibration = isotonic
   refit on *all* reviewed labels (validation already happened out-of-fold in the factory).
2. **Full-corpus pass** — LLM-extract every candidate case (checkpointed; the 120 sampled
   cases are already in the shared checkpoint and are not re-run).
3. **Deployed column** — `data/field_<name>_deployed.parquet`: label, raw + calibrated
   confidence, evidence, provenance per case; non-candidates are False/0.02.
4. **Sidecar meta** — `data/field_<name>_meta.json`: definition, lexicon, threshold,
   validation numbers. **`ask.ipynb` discovers deployed fields through these sidecars** and
   routes matching questions to the calibrated handler automatically — define → review →
   deploy → query, no further wiring.
5. **Demo** — the threshold-aware count with abstention audit, straight from the new column.

## 1. Config + gate (reuses the factory's field definition by exec)

In [2]:
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

FIELD_NAME_DEPLOY = "applicant_is_father"      # must match the factory run

# exec the factory's config/lexicon/detector cells — single source of truth
src_f = json.loads(Path("field_factory.ipynb").read_text())
_g = {"re": re, "json": json}
for marker in ['FIELD_ID = "', "LEXICON = list(SEED_TERMS)", "def detect(full_text)"]:
    cell = next("".join(c["source"]) for c in src_f["cells"]
                if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec(cell, _g)
assert _g["FIELD_NAME"] == FIELD_NAME_DEPLOY, \
    f"the factory's FIELD_ID is {_g['FIELD_NAME']!r} — set it to {FIELD_NAME_DEPLOY!r} there, "\
    f"or change FIELD_NAME_DEPLOY here"
FIELD_DEFINITION, LEXICON, LEX_RE = _g["FIELD_DEFINITION"], _g["LEXICON"], _g["LEX_RE"]
FIELD_KIND, FIELD_UNIT = _g["FIELD_KIND"], _g.get("FIELD_UNIT")
NUMERIC, CELL_KEY = _g["NUMERIC"], _g["CELL_KEY"]
detect, det = _g["detect"], _g["det"]
LABELS   = Path(f"../data/field_{FIELD_NAME_DEPLOY}_labels.csv")
CKPT     = Path(f"../data/field_{FIELD_NAME_DEPLOY}_llm_checkpoint.json")
OUT_PARQ = Path(f"../data/field_{FIELD_NAME_DEPLOY}_deployed.parquet")
OUT_META = Path(f"../data/field_{FIELD_NAME_DEPLOY}_meta.json")
THRESHOLD = 0.7
LLM_MODEL, MAX_SENTS = _g["LLM_MODEL"], _g["MAX_SENTS"]

# ---- the quality gate ----
if not LABELS.exists():
    raise SystemExit(f"GATE: no reviewed labels ({LABELS.name}). Run the factory, review "
                     f"the template, save the labels — then deploy.")
if NUMERIC:
    lab = pd.read_csv(LABELS, keep_default_na=False, dtype=str)
    _g_col = lab[f"gold_{FIELD_NAME_DEPLOY}"].str.strip().str.lower()
    lab, _g_col = lab[_g_col != ""].copy(), _g_col[_g_col != ""]
    lab["gold"] = [np.nan if v in ("none", "null", "na", "n/a", "-")
                   else pd.to_numeric(v, errors="coerce") for v in _g_col]
    for _c in ("draft_value", "draft_conf"):
        lab[_c] = pd.to_numeric(lab[_c], errors="coerce")
    both = lab.gold.notna() & lab.draft_value.notna()
    correct = (both & (lab.gold == lab.draft_value)) | (lab.gold.isna() & lab.draft_value.isna())
    flips = int((~correct).sum())
    err = (lab.gold[both] - lab.draft_value[both]).abs()
    headline = {"accuracy": round(float(correct.mean()), 3),
                "exact": round(float((err == 0).mean()), 3),
                "mae": round(float(err.mean()), 2)}
    print(f"GATE OK: {len(lab)} reviewed labels | flip rate {flips}/{len(lab)} "
          f"({flips/len(lab)*100:.0f}%) | accuracy {headline['accuracy']}, "
          f"MAE {headline['mae']} {FIELD_UNIT}")
    fit = lab.draft_value.notna()          # confidence is a claim about a value
else:
    lab = pd.read_csv(LABELS)
    gcol = f"gold_{FIELD_NAME_DEPLOY}"
    lab["gold"] = pd.to_numeric(lab[gcol], errors="coerce")
    lab = lab[lab.gold.notna()].copy()
    lab["gold"] = lab.gold.astype(int)
    correct = lab.gold == lab.draft_label.astype(int)
    flips = int((~correct).sum())
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f1, _ = precision_recall_fscore_support(lab.gold, lab.draft_label.astype(int),
                                                  average="binary", zero_division=0)
    headline = {"llm_f1": round(float(f1), 3)}
    print(f"GATE OK: {len(lab)} reviewed labels | flip rate {flips}/{len(lab)} "
          f"({flips/len(lab)*100:.0f}%) | LLM F1 vs gold {f1:.3f} (P {p:.3f} / R {r:.3f})")
    fit = pd.Series(True, index=lab.index)
if flips == 0:
    print("!! WARNING: zero flips — labels may be rubber-stamped; validation is weak.")

# deployment calibration: refit on ALL reviewed labels (validated OOF in the factory);
# the target is "this drafted cell is correct", so conf_cal reads as P(cell is right)
from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
iso.fit(lab.loc[fit, "draft_conf"].to_numpy(float), correct[fit].to_numpy().astype(float))
print(f"deployment isotonic fitted on {int(fit.sum())} reviewed labels")

field: applicant_is_father (boolean) | corpus: echr_parental_alienation.json | sample N=120 | 5 fields defined
lexicon loaded from field_applicant_is_father_lexicon.json — frozen with the sample, not re-expanded

final lexicon: 7 patterns
section splitter ready | applicant-context sections: {'FACTS', 'unparsed', 'PROCEDURE', 'HEADER'}
corpus: 1574 records
cases with >=1 mention: 720 | conf>=0.5: 510
GATE OK: 120 reviewed labels | flip rate 45/120 (38%) | LLM F1 vs gold 0.667 (P 0.789 / R 0.577)
deployment isotonic fitted on 120 reviewed labels


## 2. Full-corpus pass (checkpointed; sampled cases already cached)

In [4]:
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
except Exception as e:
    raise SystemExit(f"Ollama unreachable ({e}) — start `ollama serve` for the full pass")

if NUMERIC:
    PROMPT = ("You extract one number from excerpts of an ECHR family-law case.\n\n"
              "Definition: " + FIELD_DEFINITION +
              "\n\nSentences from the case (with document section):\n{sents}\n\n"
              "Reply ONLY with a JSON object: "
              '{{"value": <the number, or null if the case does not state one>, '
              '"confidence": 0.0-1.0, '
              '"evidence": "<best supporting sentence, verbatim>"}}')
else:
    PROMPT = ("You classify excerpts from an ECHR family-law case.\n\nDefinition: "
              + FIELD_DEFINITION +
              "\n\nSentences from the case (with document section):\n{sents}\n\n"
              "Reply ONLY with a JSON object: "
              '{{"label": true or false, "confidence": 0.0-1.0, '
              '"evidence": "<best supporting sentence, verbatim>"}}')


def _cell(obj):
    if not NUMERIC:
        return bool(obj.get("label"))
    v = obj.get("value")
    return None if v is None or v == "" else float(v)
_JSON = re.compile(r"\{.*\}", re.S)

done = json.loads(CKPT.read_text()) if CKPT.exists() else {}
candidates = [iid for iid, d in det.items() if d["pool"]]
todo = [i for i in candidates if i not in done]
print(f"candidates: {len(candidates)} | cached: {len(done)} | to run: {len(todo)}")
import time
t0 = time.time()
for n, iid in enumerate(todo, 1):
    sents = "\n".join(f"- [{lab_}] {s}" for lab_, s in det[iid]["pool"])
    out = None
    for _ in range(2):
        try:
            rr = ollama.chat(model=LLM_MODEL,
                             messages=[{"role": "user", "content": PROMPT.format(sents=sents)}],
                             format="json",            # the parser reads one JSON object; let the model emit
                                    # only that. Without it llama3.2 generated 429
                                    # tokens for an ~80-token answer (146 s of a
                                    # 3-min call spent on text we discard).
                                    options={"temperature": 0, "num_predict": 200})
            m = _JSON.search(rr["message"]["content"])
            obj = json.loads(m.group(0))
            out = {CELL_KEY: _cell(obj),
                   "conf": max(0.0, min(1.0, float(obj.get("confidence", 0.5)))),
                   "evidence": str(obj.get("evidence", ""))[:300]}
            break
        except Exception:
            pass
    done[iid] = out or {CELL_KEY: None if NUMERIC else False, "conf": 0.5,
                        "evidence": "unparseable"}
    if n % 10 == 0 or n == len(todo):
        CKPT.write_text(json.dumps(done))
        rate = (time.time() - t0) / n
        print(f"  {n}/{len(todo)}  ({rate:.1f}s/case, ~{rate*(len(todo)-n)/60:.0f} min left)")
print("full pass complete")

candidates: 720 | cached: 120 | to run: 620
  10/620  (22.5s/case, ~229 min left)
  20/620  (21.9s/case, ~219 min left)
  30/620  (23.0s/case, ~226 min left)
  40/620  (23.4s/case, ~226 min left)
  50/620  (22.9s/case, ~218 min left)
  60/620  (23.0s/case, ~214 min left)
  70/620  (22.7s/case, ~208 min left)
  80/620  (22.2s/case, ~200 min left)
  90/620  (21.5s/case, ~190 min left)
  100/620  (21.7s/case, ~188 min left)
  110/620  (21.6s/case, ~184 min left)
  120/620  (21.2s/case, ~177 min left)
  130/620  (21.2s/case, ~173 min left)
  140/620  (21.2s/case, ~169 min left)
  150/620  (21.2s/case, ~166 min left)
  160/620  (20.9s/case, ~160 min left)
  170/620  (21.0s/case, ~157 min left)
  180/620  (20.8s/case, ~152 min left)
  190/620  (20.9s/case, ~150 min left)
  200/620  (20.9s/case, ~146 min left)
  210/620  (21.0s/case, ~143 min left)
  220/620  (21.1s/case, ~140 min left)
  230/620  (21.0s/case, ~137 min left)
  240/620  (21.0s/case, ~133 min left)
  250/620  (21.1s/case, ~130 

## 3. Deployed column + sidecar meta

In [6]:
rows = []
for iid, d in det.items():
    if d["pool"] and iid in done:
        o = done[iid]
        raw = float(o["conf"])
        val = o[CELL_KEY] if NUMERIC else bool(o[CELL_KEY])
        rows.append({"id": iid, FIELD_NAME_DEPLOY: val,
                     "conf_raw": raw, "conf_cal": float(iso.predict([raw])[0]),
                     "evidence": o.get("evidence"),
                     "provenance": f"llm:{LLM_MODEL};sents={len(d['pool'])}"})
    else:
        rows.append({"id": iid, FIELD_NAME_DEPLOY: None if NUMERIC else False, "conf_raw": 0.02,
                     "conf_cal": float(iso.predict([0.02])[0]), "evidence": None,
                     "provenance": "no-lexicon-mention"})
dep = pd.DataFrame(rows)
if NUMERIC:
    dep[FIELD_NAME_DEPLOY] = pd.to_numeric(dep[FIELD_NAME_DEPLOY], errors="coerce")
dep.to_parquet(OUT_PARQ, index=False)
if NUMERIC:
    _has = dep[FIELD_NAME_DEPLOY].notna()
    print(f"wrote {OUT_PARQ.name}: {len(dep)} rows | value extracted: {int(_has.sum())} "
          f"| confident (cal>={THRESHOLD}): "
          f"{int((_has & (dep.conf_cal >= THRESHOLD)).sum())} | the rest state no value")
else:
    print(f"wrote {OUT_PARQ.name}: {len(dep)} rows | positive: {int(dep[FIELD_NAME_DEPLOY].sum())} "
          f"| confident (cal>={THRESHOLD}): {int((dep[FIELD_NAME_DEPLOY] & (dep.conf_cal >= THRESHOLD)).sum())}")

meta = {"field": FIELD_NAME_DEPLOY, "kind": FIELD_KIND, "unit": FIELD_UNIT,
        "definition": FIELD_DEFINITION,
        "lexicon": sorted(set(LEXICON)), "threshold": THRESHOLD,
        "question_terms": _g.get("QUESTION_TERMS", []),
        "corpus": "echr_parental_alienation.json (ECHR-EN; domain of validity)",
        "validation": {"n_labels": int(len(lab)), "flip_rate": f"{flips}/{len(lab)}",
                       **headline},
        "labels_file": LABELS.name, "model": LLM_MODEL,
        # who produced the gold — a reader of the column must not have to guess
        "labels_provenance": (pd.read_csv(LABELS).get("notes", pd.Series(["unrecorded"]))
                              .dropna().iloc[0] if LABELS.exists() else "unrecorded")}
OUT_META.write_text(json.dumps(meta, indent=2))
print(f"wrote {OUT_META.name} — ask.ipynb discovers the field through this sidecar")

wrote field_applicant_is_father_deployed.parquet: 1574 rows | positive: 417 | confident (cal>=0.7): 0
wrote field_applicant_is_father_meta.json — ask.ipynb discovers the field through this sidecar


## 4. Demo — the new field answering its Bucket-3 question, threshold-aware

In [8]:
import duckdb
con = duckdb.connect()
con.register("f", dep)
if NUMERIC:
    q = f"""SELECT COUNT(*) FILTER ({FIELD_NAME_DEPLOY} IS NOT NULL AND conf_cal >= {THRESHOLD}) AS confident,
                   COUNT(*) FILTER ({FIELD_NAME_DEPLOY} IS NOT NULL AND conf_cal <  {THRESHOLD}) AS abstained,
                   COUNT(*) AS corpus,
                   median({FIELD_NAME_DEPLOY}) FILTER (conf_cal >= {THRESHOLD}) AS median_value
            FROM f"""
    print(f'Q: "What {FIELD_NAME_DEPLOY.replace("_", " ")} do these cases state?"')
    print(con.execute(q).df().to_string(index=False))
    print(f"\nANSWER pattern: <confident> cases carry a calibrated value (conf >= {THRESHOLD}) "
          f"in {FIELD_UNIT}; <abstained> extracted values abstained (surfaced, not dropped); "
          f"the rest state no value.")
    print(f"CAVEAT: LLM extractor, accuracy {headline['accuracy']} / MAE {headline['mae']} "
          f"{FIELD_UNIT} on {len(lab)} reviewed labels (flip rate {flips}/{len(lab)}); "
          f"ECHR-English only.")
else:
    q = f"""SELECT COUNT(*) FILTER ({FIELD_NAME_DEPLOY} AND conf_cal >= {THRESHOLD}) AS confident,
                   COUNT(*) FILTER ({FIELD_NAME_DEPLOY} AND conf_cal <  {THRESHOLD}) AS abstained,
                   COUNT(*) AS corpus
            FROM f"""
    print(f'Q: "How many cases involve {FIELD_NAME_DEPLOY.replace("_", " ")}?"')
    print(con.execute(q).df().to_string(index=False))
    print(f"\nANSWER pattern: <confident> cases at calibrated conf >= {THRESHOLD}; "
          f"<abstained> predicted-positive cells abstained (surfaced, not dropped).")
    print(f"CAVEAT: LLM extractor, F1 {headline['llm_f1']} on {len(lab)} reviewed labels "
          f"(flip rate {flips}/{len(lab)}); ECHR-English only.")

Q: "How many cases involve applicant is father?"
 confident  abstained  corpus
         0        417    1574

ANSWER pattern: <confident> cases at calibrated conf >= 0.7; <abstained> predicted-positive cells abstained (surfaced, not dropped).
CAVEAT: LLM extractor, F1 0.667 on 120 reviewed labels (flip rate 45/120); ECHR-English only.
